<h1 align= center; style="color: white;background-color:black;
         font-size:47px; font-weight:bold;padding:20px;
    font-style:italic;border:3px  solid black;
    box-shadow:2px 2px 5px gray;text-align:center;border-radius:5px;
">   YELP REVIEW DATASET </span></h1>


<h2 style="background-color:LIGHTBLUE; color:black; text-align:center;
    font-size:30px; font-weight:bold;padding:10px;">IMPORTING LIBRARIES</h1> 

In [1]:
import pandas as pd
import string
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split

<h2 style="background-color:LIGHTBLUE; color:black; text-align:center;
    font-size:30px; font-weight:bold;padding:10px;">READING THE DATASET</h1>

In [2]:
df=pd.read_csv(r"C:\Users\USER\Downloads\yelp.csv",encoding="ISO-8859-1")
df

,business_id,date,review_id,stars,text,type,user_id,cool,useful,funny
0,9yKzy9PApeiPPOUJEtnvkg,2011-01-26,fWKvX83p0-ka4JS3dc6E5A,5,My wife took me here on my birthday for breakf...,review,rLtl8ZkDX5vH5nAx9C3q5Q,2,5,0
1,ZRJwVLyzEJq1VAihDhYiow,2011-07-27,IjZ33sJrzXqU-0X6U8NwyA,5,I have no idea why some people give bad review...,review,0a2KyEL0d3Yb1V6aivbIuQ,0,0,0
2,6oRAC4uyJCsJl1X0WZpVSA,2012-06-14,IESLBzqUCLdSzSqm0eCSxQ,4,love the gyro plate. Rice is so good and I als...,review,0hT2KtfLiobPvh6cDC8JQg,0,1,0
3,_1QQZuf4zZOyFCvXc0o6Vg,2010-05-27,G-WvGaISbqqaMHlNnByodA,5,"Rosie, Dakota, and I LOVE Chaparral Dog Park!!...",review,uZetl9T0NcROGOyFfughhg,1,2,0
4,6ozycU1RpktNG2-1BroVtw,2012-01-05,1uJFq2r5QfJG_6ExMRCaGw,5,General Manager Scott Petello is a good egg!!!...,review,vYmM4KTsC8ZfQBg-j5MWkw,0,0,0
...,...,...,...,...,...,...,...,...,...,...
9995,VY_tvNUCCXGXQeSvJl757Q,2012-07-28,Ubyfp2RSDYW0g7Mbr8N3iA,3,First visit...Had lunch here today - used my G...,review,_eqQoPtQ3e3UxLE4faT6ow,1,2,0
9996,EKzMHI1tip8rC1-ZAy64yg,2012-01-18,2XyIOQKbVFb6uXQdJ0RzlQ,4,Should be called house of deliciousness!\n\nI ...,review,ROru4uk5SaYc3rg8IU7SQw,0,0,0
9997,53YGfwmbW73JhFiemNeyzQ,2010-11-16,jyznYkIbpqVmlsZxSDSypA,4,I recently visited Olive and Ivy for business ...,review,gGbN1aKQHMgfQZkqlsuwzg,0,0,0
9998,9SKdOoDHcFoxK5ZtsgHJoA,2012-12-02,5UKq9WQE1qQbJ0DJbc-B6Q,2,My nephew just moved to Scottsdale recently so...,review,0lyVoNazXa20WzUyZPLaQQ,0,0,0


In [3]:
df = df.drop(columns=['useful', 'funny', 'cool', 'date'])

df.columns


Index(['business_id', 'review_id', 'stars', 'text', 'type', 'user_id'], dtype='object')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   business_id  10000 non-null  object
 1   review_id    10000 non-null  object
 2   stars        10000 non-null  int64 
 3   text         10000 non-null  object
 4   type         10000 non-null  object
 5   user_id      10000 non-null  object
dtypes: int64(1), object(5)
memory usage: 468.9+ KB


 <h2 style="background-color:LIGHTBLUE; color:black; text-align:center;
    font-size:30px; font-weight:bold;padding:10px;">  MAPPING STARS TO SENTIMENT</h1>

In [5]:

def get_sentiment(stars):
    if stars <= 2:
        return 'Negative'
    elif stars == 3:
        return 'Neutral'
    else:
        return 'Positive'
df['sentiment'] = df['stars'].apply(get_sentiment)

df[['text', 'stars', 'sentiment']].head()
df

,business_id,review_id,stars,text,type,user_id,sentiment
0,9yKzy9PApeiPPOUJEtnvkg,fWKvX83p0-ka4JS3dc6E5A,5,My wife took me here on my birthday for breakf...,review,rLtl8ZkDX5vH5nAx9C3q5Q,Positive
1,ZRJwVLyzEJq1VAihDhYiow,IjZ33sJrzXqU-0X6U8NwyA,5,I have no idea why some people give bad review...,review,0a2KyEL0d3Yb1V6aivbIuQ,Positive
2,6oRAC4uyJCsJl1X0WZpVSA,IESLBzqUCLdSzSqm0eCSxQ,4,love the gyro plate. Rice is so good and I als...,review,0hT2KtfLiobPvh6cDC8JQg,Positive
3,_1QQZuf4zZOyFCvXc0o6Vg,G-WvGaISbqqaMHlNnByodA,5,"Rosie, Dakota, and I LOVE Chaparral Dog Park!!...",review,uZetl9T0NcROGOyFfughhg,Positive
4,6ozycU1RpktNG2-1BroVtw,1uJFq2r5QfJG_6ExMRCaGw,5,General Manager Scott Petello is a good egg!!!...,review,vYmM4KTsC8ZfQBg-j5MWkw,Positive
...,...,...,...,...,...,...,...
9995,VY_tvNUCCXGXQeSvJl757Q,Ubyfp2RSDYW0g7Mbr8N3iA,3,First visit...Had lunch here today - used my G...,review,_eqQoPtQ3e3UxLE4faT6ow,Neutral
9996,EKzMHI1tip8rC1-ZAy64yg,2XyIOQKbVFb6uXQdJ0RzlQ,4,Should be called house of deliciousness!\n\nI ...,review,ROru4uk5SaYc3rg8IU7SQw,Positive
9997,53YGfwmbW73JhFiemNeyzQ,jyznYkIbpqVmlsZxSDSypA,4,I recently visited Olive and Ivy for business ...,review,gGbN1aKQHMgfQZkqlsuwzg,Positive
9998,9SKdOoDHcFoxK5ZtsgHJoA,5UKq9WQE1qQbJ0DJbc-B6Q,2,My nephew just moved to Scottsdale recently so...,review,0lyVoNazXa20WzUyZPLaQQ,Negative


In [6]:
def map_sentiment(stars):
    if stars <= 2:
        return -1      
    elif stars == 3:
        return 0       
    else:
        return 1       

df['sentiment'] = df['stars'].apply(map_sentiment)


In [7]:
df

,business_id,review_id,stars,text,type,user_id,sentiment
0,9yKzy9PApeiPPOUJEtnvkg,fWKvX83p0-ka4JS3dc6E5A,5,My wife took me here on my birthday for breakf...,review,rLtl8ZkDX5vH5nAx9C3q5Q,1
1,ZRJwVLyzEJq1VAihDhYiow,IjZ33sJrzXqU-0X6U8NwyA,5,I have no idea why some people give bad review...,review,0a2KyEL0d3Yb1V6aivbIuQ,1
2,6oRAC4uyJCsJl1X0WZpVSA,IESLBzqUCLdSzSqm0eCSxQ,4,love the gyro plate. Rice is so good and I als...,review,0hT2KtfLiobPvh6cDC8JQg,1
3,_1QQZuf4zZOyFCvXc0o6Vg,G-WvGaISbqqaMHlNnByodA,5,"Rosie, Dakota, and I LOVE Chaparral Dog Park!!...",review,uZetl9T0NcROGOyFfughhg,1
4,6ozycU1RpktNG2-1BroVtw,1uJFq2r5QfJG_6ExMRCaGw,5,General Manager Scott Petello is a good egg!!!...,review,vYmM4KTsC8ZfQBg-j5MWkw,1
...,...,...,...,...,...,...,...
9995,VY_tvNUCCXGXQeSvJl757Q,Ubyfp2RSDYW0g7Mbr8N3iA,3,First visit...Had lunch here today - used my G...,review,_eqQoPtQ3e3UxLE4faT6ow,0
9996,EKzMHI1tip8rC1-ZAy64yg,2XyIOQKbVFb6uXQdJ0RzlQ,4,Should be called house of deliciousness!\n\nI ...,review,ROru4uk5SaYc3rg8IU7SQw,1
9997,53YGfwmbW73JhFiemNeyzQ,jyznYkIbpqVmlsZxSDSypA,4,I recently visited Olive and Ivy for business ...,review,gGbN1aKQHMgfQZkqlsuwzg,1
9998,9SKdOoDHcFoxK5ZtsgHJoA,5UKq9WQE1qQbJ0DJbc-B6Q,2,My nephew just moved to Scottsdale recently so...,review,0lyVoNazXa20WzUyZPLaQQ,-1


In [8]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# cleaning function
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z ]', '', text)
    words = word_tokenize(text)
    words = [w for w in words if w not in stop_words]
    words = [lemmatizer.lemmatize(w) for w in words]
    return " ".join(words)

# apply cleaning
df['clean_review'] = df['text'].apply(clean_text)

# check result
print(df[['text', 'clean_review']].head())

                                                text  \
0  My wife took me here on my birthday for breakf...   
1  I have no idea why some people give bad review...   
2  love the gyro plate. Rice is so good and I als...   
3  Rosie, Dakota, and I LOVE Chaparral Dog Park!!...   
4  General Manager Scott Petello is a good egg!!!...   

                                        clean_review  
0  wife took birthday breakfast excellent weather...  
1  idea people give bad review place go show plea...  
2  love gyro plate rice good also dig candy selec...  
3  rosie dakota love chaparral dog park convenien...  
4  general manager scott petello good egg go deta...  


In [9]:

# features and target
X = df['clean_review']
y = df['sentiment']

# split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# check sizes
print("X_train size:", X_train.shape)
print("X_test size:", X_test.shape)
print("y_train size:", y_train.shape)
print("y_test size:", y_test.shape)


X_train size: (8000,)
X_test size: (2000,)
y_train size: (8000,)
y_test size: (2000,)


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

# create vectorizer
tfidf = TfidfVectorizer(max_features=5000)

# fit on training data
X_train_tfidf = tfidf.fit_transform(X_train)

# transform test data
X_test_tfidf = tfidf.transform(X_test)

# check shapes
print("X_train_tfidf shape:", X_train_tfidf.shape)
print("X_test_tfidf shape:", X_test_tfidf.shape)


X_train_tfidf shape: (8000, 5000)
X_test_tfidf shape: (2000, 5000)


In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# create model
model = LogisticRegression(max_iter=1000)

# train model
model.fit(X_train_tfidf, y_train)

# predict
y_pred = model.predict(X_test_tfidf)

# evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))



Accuracy: 0.788


In [12]:

new_review = "The food was amazing but the service was slow"
clean_new_review = clean_text(new_review)
new_review_tfidf = tfidf.transform([clean_new_review])
prediction = model.predict(new_review_tfidf)
if prediction[0] == 1:
    print("Sentiment: Positive 😊")
else:
    print("Sentiment: Negative 😞")


Sentiment: Positive 😊
